                         CLINICAL AGENT
                              │
          ┌───────────────────┼───────────────────┐
          ▼                   ▼                   ▼
     patient.csv        diagnosis.csv       pastHistory.csv
          │                   │                   │
          │                   │                   │
      age                 diagnoses          previous
      gender               0–24 h            conditions
      unit type
      admission source
      admission diagnosis


admissionDx.csv was inspected as a possible additional source for admission
diagnosis. Its main diagnosis was found to be identical to apacheadmissiondx
in patient.csv, so it was not retained.

In [1]:
import pandas as pd
import os

DATA_DIR = "data"

patient = pd.read_csv(os.path.join(DATA_DIR, "patient.csv.gz"), low_memory=False)
admissionDx = pd.read_csv(os.path.join(DATA_DIR, "admissionDx.csv.gz"), low_memory=False)


### General inspection

In [2]:
print("PATIENT DATASET")
print("Shape:", patient.shape)

print(
    "Unique ICU stays:",
    patient["patientunitstayid"].nunique()
)

print(
    "Duplicated ICU stay IDs:",
    patient["patientunitstayid"].duplicated().sum()
)

display(patient.head())

PATIENT DATASET
Shape: (2520, 29)
Unique ICU stays: 2520
Duplicated ICU stay IDs: 0


,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,141764,129391,Female,87,Caucasian,59,91,NaN,157.5,23:36:00,...,ICU to SDU,2,stepdown/other,NaN,NaN,18:58:00,344,Home,Alive,002-1039
1,141765,129391,Female,87,Caucasian,59,91,"Rhythm disturbance (atrial, supraventricular)",157.5,23:36:00,...,Emergency Department,1,admit,46.5,45.0,13:14:00,2250,Step-Down Unit (SDU),Alive,002-1039
2,143870,131022,Male,76,Caucasian,68,103,"Endarterectomy, carotid",167.0,20:46:00,...,Operating Room,1,admit,77.5,79.4,10:00:00,793,Floor,Alive,002-12289
3,144815,131736,Female,34,Caucasian,56,82,"Overdose, other toxin, poison or drug",172.7,01:44:00,...,Emergency Department,1,admit,60.3,60.7,20:48:00,1121,Other External,Alive,002-1116
4,145427,132209,Male,61,Caucasian,68,103,"GI perforation/rupture, surgery for",177.8,23:48:00,...,Operating Room,1,admit,91.7,93.1,22:47:00,1369,Floor,Alive,002-12243


### SELECT VARIABLES FOR THE CLINICAL AGENT

In [3]:
patient_cols = [
    "patientunitstayid",
    "age",
    "gender",
    "unittype",
    "unitadmitsource",
    "apacheadmissiondx"
]

clinical_patient = patient[patient_cols].copy()

display(clinical_patient.head())

,patientunitstayid,age,gender,unittype,unitadmitsource,apacheadmissiondx
0,141764,87,Female,Med-Surg ICU,ICU to SDU,NaN
1,141765,87,Female,Med-Surg ICU,Emergency Department,"Rhythm disturbance (atrial, supraventricular)"
2,143870,76,Male,SICU,Operating Room,"Endarterectomy, carotid"
3,144815,34,Female,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug"
4,145427,61,Male,SICU,Operating Room,"GI perforation/rupture, surgery for"


### missing values

In [4]:
display(
    clinical_patient
    .isna()
    .sum()
    .to_frame("missing_values")
)

,missing_values
patientunitstayid,0
age,4
gender,4
unittype,0
unitadmitsource,22
apacheadmissiondx,299


### value distribution

In [5]:
for col in [
    "age",
    "gender",
    "unittype",
    "unitadmitsource",
    "apacheadmissiondx"
]:
    print(f"\n--- {col} ---")
    print(
        clinical_patient[col]
        .value_counts(dropna=False)
        .head(5)
    )


--- age ---
age
> 89    98
78      76
68      70
67      70
79      66
Name: count, dtype: int64

--- gender ---
gender
Male      1508
Female    1008
NaN          4
Name: count, dtype: int64

--- unittype ---
unittype
Med-Surg ICU    1898
MICU             142
Cardiac ICU      133
SICU              83
CCU-CTICU         82
Name: count, dtype: int64

--- unitadmitsource ---
unitadmitsource
Emergency Department    1304
Floor                    330
Operating Room           253
ICU to SDU               148
Direct Admit             141
Name: count, dtype: int64

--- apacheadmissiondx ---
apacheadmissiondx
NaN                                              299
Sepsis, pulmonary                                121
Diabetic ketoacidosis                            105
CHF, congestive heart failure                    102
Rhythm disturbance (atrial, supraventricular)     79
Name: count, dtype: int64


heck whether admissionDx adds information beyond apacheadmissiondx

In [6]:
admissionDx_main = admissionDx[
    admissionDx["admitdxpath"].str.contains(
        "All Diagnosis",
        case=False,
        na=False
    )
][
    ["patientunitstayid", "admitdxname"]
].copy()

comparison = clinical_patient[
    ["patientunitstayid", "apacheadmissiondx"]
].merge(
    admissionDx_main,
    on="patientunitstayid",
    how="left"
)

same_diagnosis = (
    comparison["apacheadmissiondx"]
    .fillna("")
    .str.strip()
    ==
    comparison["admitdxname"]
    .fillna("")
    .str.strip()
)

print("Patients:", len(comparison))
print("Matching admission diagnoses:", same_diagnosis.sum())
print(
    "Match percentage:",
    round(same_diagnosis.mean() * 100, 2),
    "%"
)

Patients: 2520
Matching admission diagnoses: 2520
Match percentage: 100.0 %


`admissionDx.csv` was evaluated as an additional source of admission diagnosis.

The main diagnosis extracted from `admissionDx.csv` matched
`apacheadmissiondx` in `patient.csv` for 100% of ICU stays.

Therefore, `admissionDx.csv` was considered redundant and excluded
from further preprocessing.

### clean patient.csv variables

In [7]:
clinical_patient_clean = clinical_patient.copy()

# Replace "> 89" with 90 so age can be represented numerically
clinical_patient_clean["age"] = (
    clinical_patient_clean["age"]
    .replace("> 89", "90")
)

clinical_patient_clean["age"] = pd.to_numeric(
    clinical_patient_clean["age"],
    errors="coerce"
)

clinical_patient_clean["age"].describe()

count    2516.000000
mean       63.281002
std        17.764847
min        15.000000
25%        53.000000
50%        66.000000
75%        77.000000
max        90.000000
Name: age, dtype: float64

In [8]:
# Clean string variables
string_cols = [
    "gender",
    "unittype",
    "unitadmitsource",
    "apacheadmissiondx"
]

for col in string_cols:
    clinical_patient_clean[col] = (
        clinical_patient_clean[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA
        })
    )
    
clinical_patient_clean = clinical_patient_clean.rename(columns={"unittype": "unit_type",
    "unitadmitsource": "admission_source", "apacheadmissiondx": "admission_diagnosis"
})

display(clinical_patient_clean.head())

,patientunitstayid,age,gender,unit_type,admission_source,admission_diagnosis
0,141764,87.0,Female,Med-Surg ICU,ICU to SDU,<NA>
1,141765,87.0,Female,Med-Surg ICU,Emergency Department,"Rhythm disturbance (atrial, supraventricular)"
2,143870,76.0,Male,SICU,Operating Room,"Endarterectomy, carotid"
3,144815,34.0,Female,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug"
4,145427,61.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for"


In [9]:
# check the cleaned patient.csv
print("Shape:", clinical_patient_clean.shape)

print("\nMissing values:")
display(
    clinical_patient_clean
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nAge summary:")
display(clinical_patient_clean["age"].describe())

print("\nGender:")
print(clinical_patient_clean["gender"].value_counts(dropna=False))

print("\nUnit type:")
print(clinical_patient_clean["unit_type"].value_counts(dropna=False))

print("\nAdmission source:")
print(clinical_patient_clean["admission_source"].value_counts(dropna=False))

Shape: (2520, 6)

Missing values:


,missing_values
patientunitstayid,0
age,4
gender,4
unit_type,0
admission_source,22
admission_diagnosis,299



Age summary:


count    2516.000000
mean       63.281002
std        17.764847
min        15.000000
25%        53.000000
50%        66.000000
75%        77.000000
max        90.000000
Name: age, dtype: float64


Gender:
gender
Male      1508
Female    1008
<NA>         4
Name: count, dtype: Int64

Unit type:
unit_type
Med-Surg ICU    1898
MICU             142
Cardiac ICU      133
SICU              83
CCU-CTICU         82
Neuro ICU         65
CSICU             65
CTICU             52
Name: count, dtype: Int64

Admission source:
admission_source
Emergency Department    1304
Floor                    330
Operating Room           253
ICU to SDU               148
Direct Admit             141
Recovery Room             76
Step-Down Unit (SDU)      65
Acute Care/Floor          56
ICU                       52
Other ICU                 31
Other Hospital            25
<NA>                      22
PACU                      15
Chest Pain Center          2
Name: count, dtype: Int64


### clean diagnosis.csv

In [10]:
diagnosis = pd.read_csv(
    os.path.join(DATA_DIR, "diagnosis.csv.gz"),
    low_memory=False
)

print("DIAGNOSIS DATASET")
print("Shape:", diagnosis.shape)
print("Columns:", diagnosis.columns.tolist())

display(diagnosis.head())

DIAGNOSIS DATASET
Shape: (24978, 7)
Columns: ['diagnosisid', 'patientunitstayid', 'activeupondischarge', 'diagnosisoffset', 'diagnosisstring', 'icd9code', 'diagnosispriority']


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,7607199,346380,False,5028,cardiovascular|ventricular disorders|hypertension,"401.9, I10",Other
1,7570429,346380,False,685,neurologic|altered mental status / pain|change...,"780.09, R41.82",Major
2,7705483,346380,True,5035,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major
3,7848601,346380,True,5035,neurologic|altered mental status / pain|schizo...,"295.90, F20.9",Major
4,7451475,346380,False,5028,pulmonary|disorders of vasculature|pulmonary e...,"415.19, I26.99",Major


In [11]:
display(
    diagnosis[
        [
            "patientunitstayid",
            "diagnosisoffset",
            "diagnosisstring"
        ]
    ].head(20)
)

print("\nMissing values:")
display(
    diagnosis[
        [
            "patientunitstayid",
            "diagnosisoffset",
            "diagnosisstring"
        ]
    ]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nDiagnosis offset summary:")
display(diagnosis["diagnosisoffset"].describe())

,patientunitstayid,diagnosisoffset,diagnosisstring
0,346380,5028,cardiovascular|ventricular disorders|hypertension
1,346380,685,neurologic|altered mental status / pain|change...
2,346380,5035,cardiovascular|shock / hypotension|hypotension
3,346380,5035,neurologic|altered mental status / pain|schizo...
4,346380,5028,pulmonary|disorders of vasculature|pulmonary e...
5,346380,5028,cardiovascular|shock / hypotension|hypotension
6,346380,5035,renal|disorder of kidney|acute renal failure
7,346380,5035,general|other syndromes|syncope
8,346380,5028,neurologic|misc|Parkinson's disease
9,346380,5028,cardiovascular|shock / hypotension|dehydration



Missing values:


,missing_values
patientunitstayid,0
diagnosisoffset,0
diagnosisstring,0



Diagnosis offset summary:


count    24978.000000
mean      3716.212907
std       5712.509594
min      -1933.000000
25%        151.000000
50%       1342.500000
75%       4558.500000
max      52277.000000
Name: diagnosisoffset, dtype: float64

In [12]:
# KEEP DIAGNOSES FROM THE FIRST 24 HOURS
diagnosis_24h = diagnosis[
    (diagnosis["diagnosisoffset"] >= 0) &
    (diagnosis["diagnosisoffset"] <= 1440)
].copy()

print("All diagnosis rows:", len(diagnosis))
print("Diagnosis rows in first 24h:", len(diagnosis_24h))

print(
    "Patients with diagnosis in first 24h:",
    diagnosis_24h["patientunitstayid"].nunique()
)

All diagnosis rows: 24978
Diagnosis rows in first 24h: 12649
Patients with diagnosis in first 24h: 2133


In [13]:
diagnosis_24h["diagnosisstring"] = (
    diagnosis_24h["diagnosisstring"]
    .astype("string")
    .str.strip()
)

diagnosis_24h = diagnosis_24h.dropna(
    subset=["diagnosisstring"]
)


# remove duplicates
diagnosis_24h = diagnosis_24h.drop_duplicates(
    subset=[
        "patientunitstayid",
        "diagnosisstring"
    ]
)

print("Rows after removing duplicates:", len(diagnosis_24h))

Rows after removing duplicates: 7506


In [14]:
# AGGREGATE 0-24H DIAGNOSES PER ICU STAY

diagnoses_24h_grouped = (
    diagnosis_24h
    .sort_values(
        ["patientunitstayid", "diagnosisoffset"]
    )
    .groupby("patientunitstayid")["diagnosisstring"]
    .apply(lambda x: list(pd.unique(x)))
    .reset_index()
    .rename(columns={
        "diagnosisstring": "diagnoses_0_24h"
    })
)

display(diagnoses_24h_grouped.head(10))

,patientunitstayid,diagnoses_0_24h
0,143870,[cardiovascular|chest pain / ASHD|coronary art...
1,145427,[gastrointestinal|post-GI surgery|s/p explorat...
2,151179,"[oncology|chest tumors|metastatic lung CA, car..."
3,151867,[gastrointestinal|intestinal disease|viscus pe...
4,151900,[pulmonary|respiratory failure|acute respirato...
5,152954,[cardiovascular|ventricular disorders|congesti...
6,153972,[cardiovascular|shock / hypotension|hypotensio...
7,155961,[cardiovascular|ventricular disorders|hyperten...
8,156308,"[pulmonary|disorders of the airways|COPD, pulm..."
9,157016,"[surgery|obstetrics|vaginal delivery, gastroin..."


### clean pastHistory.csv

In [15]:
# LOAD PAST HISTORY DATA

past_history_raw = pd.read_csv(
    os.path.join(DATA_DIR, "pastHistory.csv.gz"),
    low_memory=False
)

print("PAST HISTORY DATASET")
print("Shape:", past_history_raw.shape)
print("Columns:", past_history_raw.columns.tolist())
print("Missing values:")
display(past_history_raw.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

print("Unique patients:", past_history_raw["patientunitstayid"].nunique())

display(past_history_raw.head())

PAST HISTORY DATASET
Shape: (12109, 8)
Columns: ['pasthistoryid', 'patientunitstayid', 'pasthistoryoffset', 'pasthistoryenteredoffset', 'pasthistorynotetype', 'pasthistorypath', 'pasthistoryvalue', 'pasthistoryvaluetext']
Missing values:


,missing_values
pasthistoryid,0
patientunitstayid,0
pasthistoryoffset,0
pasthistoryenteredoffset,0
pasthistorynotetype,0
pasthistorypath,0
pasthistoryvalue,0
pasthistoryvaluetext,0


Unique patients: 2211


,pasthistoryid,patientunitstayid,pasthistoryoffset,pasthistoryenteredoffset,pasthistorynotetype,pasthistorypath,pasthistoryvalue,pasthistoryvaluetext
0,990803,141765,7,12,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
1,970059,143870,4,10,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
2,1180401,144815,32,41,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
3,1194998,145427,8,13,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems
4,896652,147307,53,56,Comprehensive Progress,notes/Progress Notes/Past History/Past History...,No Health Problems,NoHealthProblems


In [16]:
# Comparar as duas representações de past history
pd.set_option("display.max_colwidth", None)
display(
    past_history_raw[
        [
            "patientunitstayid",
            "pasthistorypath",
            "pasthistoryvalue",
            "pasthistoryvaluetext"
        ]
    ].head(30)
)

print("\nTop values - pasthistoryvalue")
print(
    past_history_raw["pasthistoryvalue"]
    .value_counts()
    .head(30)
)

print("\nTop values - pasthistoryvaluetext")
print(
    past_history_raw["pasthistoryvaluetext"]
    .value_counts()
    .head(30)
)

,patientunitstayid,pasthistorypath,pasthistoryvalue,pasthistoryvaluetext
0,141765,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
1,143870,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
2,144815,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
3,145427,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
4,147307,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
5,147784,notes/Progress Notes/Past History/Past History Obtain Options/Performed,Performed,Performed
6,147784,notes/Progress Notes/Past History/Organ Systems/Endocrine (R)/Non-Insulin Dependent Diabetes/medication dependent,medication dependent,medication dependent
7,148611,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
8,149713,notes/Progress Notes/Past History/Past History Obtain Options/No Health Problems,No Health Problems,NoHealthProblems
9,151179,notes/Progress Notes/Past History/Past History Obtain Options/Performed,Performed,Performed



Top values - pasthistoryvalue
pasthistoryvalue
Performed                                            2998
hypertension requiring treatment                     1746
insulin dependent diabetes                            457
medication dependent                                  397
CHF                                                   392
atrial fibrillation - chronic                         292
COPD  - moderate                                      276
hypothyroidism                                        273
No Health Problems                                    239
asthma                                                205
MI - date unknown                                     204
stroke - date unknown                                 189
COPD  - severe                                        177
COPD  - no limitations                                174
peripheral vascular disease                           159
procedural coronary intervention - date unknown       153
CABG - date unknown     

In [17]:
print(
    past_history_raw["pasthistorynotetype"]
    .value_counts(dropna=False)
)

pasthistorynotetype
Admission                       7096
Comprehensive Progress          3691
Past History Edit                686
Re-Admission                     379
Follow-up Consultation/Other     192
Initial Consultation/Other        65
Name: count, dtype: int64


**Past history variable selection**

patientunitstayid is retained to identify each ICU stay. pasthistoryid is only a row identifier and is therefore not clinically relevant. The offset variables are not used as Clinical Agent features, although pasthistoryenteredoffset may be used to enforce the temporal data window. pasthistorynotetype describes the source note rather than the patient's clinical history and is therefore excluded.  

pasthistoryvalue and pasthistorypath are retained because they contain complementary information: the value provides the clinical detail, while the path provides the necessary context for ambiguous entries. pasthistoryvaluetext is excluded because it is essentially redundant with pasthistoryvalue.

In [18]:
# KEEP PAST HISTORY AVAILABLE IN THE FIRST 24 HOURS

past_history_24h = past_history_raw[
    (past_history_raw["pasthistoryenteredoffset"] <= 1440)
].copy()

print("All past history rows:", len(past_history_raw))
print("Past history rows available in first 24h:", len(past_history_24h))

print(
    "Patients with past history available in first 24h:",
    past_history_24h["patientunitstayid"].nunique()
)

All past history rows: 12109
Past history rows available in first 24h: 9925
Patients with past history available in first 24h: 2208


In [19]:
# CREATE READABLE PAST HISTORY TOPICS

past_history = past_history_24h[["patientunitstayid","pasthistorypath","pasthistoryvalue"]].copy()


#remove rows where pasthistoryvalue is "performed" (case insensitive)
past_history = past_history[
    past_history["pasthistoryvalue"].str.strip().str.lower() != "performed"
].copy()

# Function to format past history topics into a more readable form
def format_past_history(path, value):

    parts = [part.strip() for part in path.split("/")]

    # Explicitly recorded absence of health problems
    if value.strip().lower() == "no health problems":
        return "No Health Problems"

    # Find the clinically relevant part of the path
    if "Organ Systems" in parts:
        idx = parts.index("Organ Systems")
        clinical_parts = parts[idx + 1:]

        # Remove "(R)" used internally in the eICU hierarchy
        clinical_parts = [
            part.replace(" (R)", "").strip()
            for part in clinical_parts
        ]

        # The final path element is usually the same as pasthistoryvalue,
        # so avoid repeating it
        if (
            clinical_parts
            and clinical_parts[-1].strip().lower()
            == value.strip().lower()
        ):
            clinical_parts = clinical_parts[:-1]

        if clinical_parts:
            organ_system = clinical_parts[0]
            details = clinical_parts[1:]

            if details:
                return (
                    f"{organ_system}: "
                    + " - ".join(details + [value.strip()])
                )

            return f"{organ_system}: {value.strip()}"

    # Fallback if the path does not follow the expected hierarchy
    return value.strip()

In [20]:
past_history["past_history_topic"] = past_history.apply(
    lambda row: format_past_history(
        row["pasthistorypath"],
        row["pasthistoryvalue"]
    ),
    axis=1
)

display(
    past_history[
        [
            "patientunitstayid",
            "pasthistoryvalue",
            "past_history_topic"
        ]
    ].head(30)
)

,patientunitstayid,pasthistoryvalue,past_history_topic
0,141765,No Health Problems,No Health Problems
1,143870,No Health Problems,No Health Problems
2,144815,No Health Problems,No Health Problems
3,145427,No Health Problems,No Health Problems
4,147307,No Health Problems,No Health Problems
6,147784,medication dependent,Endocrine: Non-Insulin Dependent Diabetes - medication dependent
7,148611,No Health Problems,No Health Problems
8,149713,No Health Problems,No Health Problems
10,151179,lung,Hematology: Oncology - Cancer - Metastases - lung
11,151179,other,Hematology: Oncology - Cancer - Cancer-Primary Site - other


In [21]:
# CLEAN REDUNDANT PAST HISTORY TOPICS

def clean_past_history_topic(topic):
    parts = [part.strip() for part in topic.split(": ", 1)]

    # Cases like "No Health Problems"
    if len(parts) == 1:
        return topic.strip()

    system, details = parts

    detail_parts = [p.strip() for p in details.split(" - ")]

    # Remove consecutive duplicates, case-insensitive
    cleaned_parts = []
    for part in detail_parts:
        if not cleaned_parts or part.lower() != cleaned_parts[-1].lower():
            cleaned_parts.append(part)

    # Remove cases where the whole final value repeats the previous category
    if len(cleaned_parts) >= 2:
        if cleaned_parts[-1].lower() == cleaned_parts[-2].lower():
            cleaned_parts = cleaned_parts[:-1]

    return f"{system}: " + " - ".join(cleaned_parts)


past_history["past_history_topic"] = (
    past_history["past_history_topic"]
    .apply(clean_past_history_topic)
)

display(past_history[[ "patientunitstayid", "pasthistoryvalue", "past_history_topic" ]].head(40))

,patientunitstayid,pasthistoryvalue,past_history_topic
0,141765,No Health Problems,No Health Problems
1,143870,No Health Problems,No Health Problems
2,144815,No Health Problems,No Health Problems
3,145427,No Health Problems,No Health Problems
4,147307,No Health Problems,No Health Problems
6,147784,medication dependent,Endocrine: Non-Insulin Dependent Diabetes - medication dependent
7,148611,No Health Problems,No Health Problems
8,149713,No Health Problems,No Health Problems
10,151179,lung,Hematology: Oncology - Cancer - Metastases - lung
11,151179,other,Hematology: Oncology - Cancer - Cancer-Primary Site - other


In [22]:
# REMOVE DUPLICATED PAST HISTORY TOPICS

past_history = past_history.drop_duplicates(
    subset=[
        "patientunitstayid",
        "past_history_topic"
    ]
)

# AGGREGATE PAST HISTORY PER ICU STAY

past_history_grouped = (
    past_history
    .groupby("patientunitstayid")["past_history_topic"]
    .apply(list)
    .reset_index()
    .rename(columns={
        "past_history_topic": "past_history"
    })
)

display(past_history_grouped.head(20))

,patientunitstayid,past_history
0,141765,[No Health Problems]
1,143870,[No Health Problems]
2,144815,[No Health Problems]
3,145427,[No Health Problems]
4,147307,[No Health Problems]
5,147784,[Endocrine: Non-Insulin Dependent Diabetes - medication dependent]
6,148611,[No Health Problems]
7,149713,[No Health Problems]
8,151179,"[Hematology: Oncology - Cancer - Metastases - lung, Hematology: Oncology - Cancer - Cancer-Primary Site - other]"
9,151867,[No Health Problems]


In [23]:
print("Patients with past history:", len(past_history_grouped))

print(
    "Unique patients:",
    past_history_grouped["patientunitstayid"].nunique()
)

print(
    "Duplicated patient IDs:",
    past_history_grouped["patientunitstayid"].duplicated().sum()
)

past_history_grouped["n_past_history"] = (
    past_history_grouped["past_history"].apply(len)
)

display(
    past_history_grouped["n_past_history"].describe()
)

Patients with past history: 2046
Unique patients: 2046
Duplicated patient IDs: 0


count    2046.00000
mean        2.99218
std         2.11247
min         1.00000
25%         1.00000
50%         2.00000
75%         4.00000
max        14.00000
Name: n_past_history, dtype: float64

## Final Merge

In [24]:
# MERGE ALL CLINICAL DATA

clinical_data = (
    clinical_patient_clean
    .merge(
        diagnoses_24h_grouped[
            ["patientunitstayid", "diagnoses_0_24h"]
        ],
        on="patientunitstayid",
        how="left"
    )
    .merge(
        past_history_grouped[
            ["patientunitstayid", "past_history"]
        ],
        on="patientunitstayid",
        how="left"
    )
)

print("Final shape:", clinical_data.shape)

display(clinical_data.head(10))

Final shape: (2520, 8)


,patientunitstayid,age,gender,unit_type,admission_source,admission_diagnosis,diagnoses_0_24h,past_history
0,141764,87.0,Female,Med-Surg ICU,ICU to SDU,<NA>,NaN,NaN
1,141765,87.0,Female,Med-Surg ICU,Emergency Department,"Rhythm disturbance (atrial, supraventricular)",NaN,[No Health Problems]
2,143870,76.0,Male,SICU,Operating Room,"Endarterectomy, carotid","[cardiovascular|chest pain / ASHD|coronary artery disease, cardiovascular|post vascular surgery|s/p cartoid endarterectomy, cardiovascular|arrhythmias|bradycardia]",[No Health Problems]
3,144815,34.0,Female,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug",NaN,[No Health Problems]
4,145427,61.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for","[gastrointestinal|post-GI surgery|s/p exploratory laparotomy, infectious diseases|GI infections|enteritis|diverticulitis of colon, infectious diseases|systemic/other infections|sepsis, gastrointestinal|post-GI surgery|s/p exploratory laparoscopy, gastrointestinal|abdominal/ general|peritonitis|due to bowel perforation]",[No Health Problems]
5,147306,55.0,Female,Med-Surg ICU,ICU to SDU,<NA>,NaN,NaN
6,147307,55.0,Female,Med-Surg ICU,Operating Room,"Endarterectomy, carotid",NaN,[No Health Problems]
7,147784,60.0,Female,Med-Surg ICU,Emergency Department,"Coma/change in level of consciousness (for hepatic see GI, for diabetic see Endocrine, if related to cardiac arrest, see CV)",NaN,[Endocrine: Non-Insulin Dependent Diabetes - medication dependent]
8,148611,28.0,Male,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug",NaN,[No Health Problems]
9,149432,34.0,Female,SICU,ICU to SDU,<NA>,NaN,NaN


In [25]:
# HANDLE MISSING DIAGNOSIS / PAST HISTORY LISTS

clinical_data["diagnoses_0_24h"] = (
    clinical_data["diagnoses_0_24h"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

clinical_data["past_history"] = (
    clinical_data["past_history"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

In [26]:
# FINAL DATASET VALIDATION

print("Rows:", len(clinical_data))

print(
    "Unique ICU stays:",
    clinical_data["patientunitstayid"].nunique()
)

print(
    "Duplicated ICU stay IDs:",
    clinical_data["patientunitstayid"].duplicated().sum()
)

print("\nColumns:")
print(clinical_data.columns.tolist())

print("\nMissing values:")
display(
    clinical_data
    .isna()
    .sum()
    .to_frame("missing_values")
)

Rows: 2520
Unique ICU stays: 2520
Duplicated ICU stay IDs: 0

Columns:
['patientunitstayid', 'age', 'gender', 'unit_type', 'admission_source', 'admission_diagnosis', 'diagnoses_0_24h', 'past_history']

Missing values:


,missing_values
patientunitstayid,0
age,4
gender,4
unit_type,0
admission_source,22
admission_diagnosis,299
diagnoses_0_24h,0
past_history,0


In [27]:
# Show patients with both diagnoses and past history

complete_cases = clinical_data[
    clinical_data["diagnoses_0_24h"].apply(len).gt(0) &
    clinical_data["past_history"].apply(len).gt(0)
]

display(
    complete_cases[
        [
            "patientunitstayid",
            "age",
            "gender",
            "unit_type",
            "admission_source",
            "admission_diagnosis",
            "diagnoses_0_24h",
            "past_history"
        ]
    ].head(10)
)

,patientunitstayid,age,gender,unit_type,admission_source,admission_diagnosis,diagnoses_0_24h,past_history
2,143870,76.0,Male,SICU,Operating Room,"Endarterectomy, carotid","[cardiovascular|chest pain / ASHD|coronary artery disease, cardiovascular|post vascular surgery|s/p cartoid endarterectomy, cardiovascular|arrhythmias|bradycardia]",[No Health Problems]
4,145427,61.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for","[gastrointestinal|post-GI surgery|s/p exploratory laparotomy, infectious diseases|GI infections|enteritis|diverticulitis of colon, infectious diseases|systemic/other infections|sepsis, gastrointestinal|post-GI surgery|s/p exploratory laparoscopy, gastrointestinal|abdominal/ general|peritonitis|due to bowel perforation]",[No Health Problems]
13,151179,59.0,Female,Med-Surg ICU,Emergency Department,"Sepsis, cutaneous/soft tissue","[oncology|chest tumors|metastatic lung CA, cardiovascular|shock / hypotension|septic shock, infectious diseases|skin, bone and joint infections|infected pressure ulcer, renal|disorder of acid base|metabolic acidosis, infectious diseases|skin, bone and joint infections|infectious dermatitis]","[Hematology: Oncology - Cancer - Metastases - lung, Hematology: Oncology - Cancer - Cancer-Primary Site - other]"
14,151867,44.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for","[gastrointestinal|intestinal disease|viscus perforation, gastrointestinal|post-GI surgery|s/p exploratory laparotomy]",[No Health Problems]
15,151900,66.0,Female,MICU,Emergency Department,"Sepsis, pulmonary","[pulmonary|respiratory failure|acute respiratory distress, endocrine|glucose metabolism|hyperglycemia, renal|disorder of kidney|acute renal failure, pulmonary|disorders of the airways|COPD, oncology|head and neck tumors|neck tumor|laryngeal CA, pulmonary|pulmonary infections|pneumonia, cardiovascular|shock / hypotension|septic shock, hematology|bleeding and red blood cell disorders|anemia, cardiovascular|chest pain / ASHD|acute coronary syndrome, hematology|platelet disorders|thrombocytosis]","[Hematology: Oncology - Cancer Therapy - Chemotherapy - chemotherapy within past mo., Pulmonary: COPD - no limitations, Hematology: Oncology - Cancer - Cancer-Primary Site - head and neck, Hematology: Oncology - Cancer Therapy - Radiation Therapy within past 6 months - primary site]"
16,152954,41.0,Female,Med-Surg ICU,Emergency Department,"Respiratory - medical, other","[cardiovascular|ventricular disorders|congestive heart failure, pulmonary|respiratory failure|acute respiratory distress, pulmonary|respiratory failure|hypoxemia, cardiovascular|shock / hypotension|signs and symptoms of sepsis (SIRS), pulmonary|disorders of lung parenchyma|interstitial lung disease|hypersensitivity pneumonitis, pulmonary|respiratory failure|acute respiratory failure, pulmonary|respiratory failure|ARDS]",[No Health Problems]
17,153972,63.0,Male,Med-Surg ICU,Emergency Department,"Bleeding, lower GI","[cardiovascular|shock / hypotension|hypotension, cardiovascular|chest pain / ASHD|coronary artery disease|s/p CABG, cardiovascular|arrhythmias|atrial fibrillation, pulmonary|disorders of the airways|obstructive sleep apnea, hematology|bleeding and red blood cell disorders|anemia, cardiovascular|ventricular disorders|cardiomyopathy|dilated|ischemic, gastrointestinal|GI bleeding / PUD|GI bleeding, hematology|coagulation disorders|coagulopathy|coumadin administration]","[Cardiovascular: Coronary Artery Bypass - CABG - remote, Cardiovascular: Arrhythmias - atrial fibrillation - chronic, Cardiovascular: Hypertension Requiring Treatment, Cardiovascular: AICD, Cardiovascular: Congestive Heart Failure - CHF - severity unknown, Cardiovascular: Valve disease - AR, Cardiovascular: Peripheral Vascular Disease, Cardiovascular: Myocardial Infarction - MI - date unknown]"
18,155961,57.0,Female,Med-Surg ICU,Operating Room,"Knee replacement, total (non-traumatic)","[cardiovascular|ventricular disorders|hypertension, pulmonary|disorders of the a

In [28]:
# DATA COVERAGE

n = len(clinical_data)

coverage = pd.DataFrame({
    "variable": [
        "age",
        "gender",
        "unit_type",
        "admission_source",
        "admission_diagnosis",
        "diagnoses_0_24h",
        "past_history"
    ],
    "available": [
        clinical_data["age"].notna().sum(),
        clinical_data["gender"].notna().sum(),
        clinical_data["unit_type"].notna().sum(),
        clinical_data["admission_source"].notna().sum(),
        clinical_data["admission_diagnosis"].notna().sum(),
        clinical_data["diagnoses_0_24h"].apply(len).gt(0).sum(),
        clinical_data["past_history"].apply(len).gt(0).sum()
    ]
})

coverage["percentage"] = (
    coverage["available"] / n * 100
).round(1)

display(coverage)

,variable,available,percentage
0,age,2516,99.8
1,gender,2516,99.8
2,unit_type,2520,100.0
3,admission_source,2498,99.1
4,admission_diagnosis,2221,88.1
5,diagnoses_0_24h,2133,84.6
6,past_history,2046,81.2


In [29]:
# PREPARE FINAL DATASET FOR EXPORT

clinical_data_export = clinical_data.copy()

# Convert lists to readable strings for CSV storage
clinical_data_export["diagnoses_0_24h"] = (
    clinical_data_export["diagnoses_0_24h"]
    .apply(lambda x: " ; ".join(x))
)

clinical_data_export["past_history"] = (
    clinical_data_export["past_history"]
    .apply(lambda x: " ; ".join(x))
)

display(clinical_data_export.head())

,patientunitstayid,age,gender,unit_type,admission_source,admission_diagnosis,diagnoses_0_24h,past_history
0,141764,87.0,Female,Med-Surg ICU,ICU to SDU,<NA>,,
1,141765,87.0,Female,Med-Surg ICU,Emergency Department,"Rhythm disturbance (atrial, supraventricular)",,No Health Problems
2,143870,76.0,Male,SICU,Operating Room,"Endarterectomy, carotid",cardiovascular|chest pain / ASHD|coronary artery disease ; cardiovascular|post vascular surgery|s/p cartoid endarterectomy ; cardiovascular|arrhythmias|bradycardia,No Health Problems
3,144815,34.0,Female,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug",,No Health Problems
4,145427,61.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for",gastrointestinal|post-GI surgery|s/p exploratory laparotomy ; infectious diseases|GI infections|enteritis|diverticulitis of colon ; infectious diseases|systemic/other infections|sepsis ; gastrointestinal|post-GI surgery|s/p exploratory laparoscopy ; gastrointestinal|abdominal/ general|peritonitis|due to bowel perforation,No Health Problems


In [30]:
# SAVE FINAL CLINICAL AGENT DATASET

OUTPUT_DIR = "processed_data"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

output_path = os.path.join(
    OUTPUT_DIR,
    "clinical_agent_data.csv"
)

clinical_data_export.to_csv(
    output_path,
    index=False
)

print("Clinical Agent dataset saved to:")
print(output_path)

Clinical Agent dataset saved to:
processed_data\clinical_agent_data.csv


In [31]:
# VERIFY SAVED DATASET

clinical_check = pd.read_csv(output_path)

print("Shape:", clinical_check.shape)

print(
    "Unique ICU stays:",
    clinical_check["patientunitstayid"].nunique()
)

print(
    "Duplicated ICU stay IDs:",
    clinical_check["patientunitstayid"].duplicated().sum()
)

display(clinical_check.head())

Shape: (2520, 8)
Unique ICU stays: 2520
Duplicated ICU stay IDs: 0


,patientunitstayid,age,gender,unit_type,admission_source,admission_diagnosis,diagnoses_0_24h,past_history
0,141764,87.0,Female,Med-Surg ICU,ICU to SDU,NaN,NaN,NaN
1,141765,87.0,Female,Med-Surg ICU,Emergency Department,"Rhythm disturbance (atrial, supraventricular)",NaN,No Health Problems
2,143870,76.0,Male,SICU,Operating Room,"Endarterectomy, carotid",cardiovascular|chest pain / ASHD|coronary artery disease ; cardiovascular|post vascular surgery|s/p cartoid endarterectomy ; cardiovascular|arrhythmias|bradycardia,No Health Problems
3,144815,34.0,Female,Med-Surg ICU,Emergency Department,"Overdose, other toxin, poison or drug",NaN,No Health Problems
4,145427,61.0,Male,SICU,Operating Room,"GI perforation/rupture, surgery for",gastrointestinal|post-GI surgery|s/p exploratory laparotomy ; infectious diseases|GI infections|enteritis|diverticulitis of colon ; infectious diseases|systemic/other infections|sepsis ; gastrointestinal|post-GI surgery|s/p exploratory laparoscopy ; gastrointestinal|abdominal/ general|peritonitis|due to bowel perforation,No Health Problems
